In [1]:
# Step 1: Setup and Imports
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
# Step 1A: Load the Data

data = pd.read_csv("../data/data.csv")

print("✅ Files loaded successfully.")
print("Data shape:", data.shape)

✅ Files loaded successfully.
Data shape: (1596, 5)


In [3]:
# Quick look
display(data.head(5))

print("\n🚫 Missing values summary:")
print(data.isna().sum())

print("\nNumber of unique countries:", data['Country Name'].nunique())
print("Number of unique indicators:", data['Series Name'].nunique())

,Country Name,Country Code,Series Name,Series Code,2022 [YR2022]
0,Afghanistan,AFG,Carbon dioxide (CO2) emissions excluding LULUC...,EN.GHG.CO2.PC.CE.AR5,0.20355189
1,Afghanistan,AFG,GDP per capita (constant 2015 US$),NY.GDP.PCAP.KD,377.6656271
2,Afghanistan,AFG,Energy use (kg of oil equivalent per capita),EG.USE.PCAP.KG.OE,..
3,Afghanistan,AFG,Urban population (% of total population),SP.URB.TOTL.IN.ZS,26.616
4,Afghanistan,AFG,Trade (% of GDP),NE.TRD.GNFS.ZS,72.88546961



🚫 Missing values summary:
Country Name     0
Country Code     0
Series Name      0
Series Code      0
2022 [YR2022]    0
dtype: int64

Number of unique countries: 266
Number of unique indicators: 6


In [4]:
# Step 2A: Replace ".." and similar placeholders with NaN
data['2022 [YR2022]'] = data['2022 [YR2022]'].replace("..", np.nan)

# Convert to numeric (coerce anything non-numeric)
data['2022 [YR2022]'] = pd.to_numeric(data['2022 [YR2022]'], errors='coerce')

# Verify cleaning
print("Unique value types:", data['2022 [YR2022]'].apply(type).unique())
print("Number of numeric entries:", data['2022 [YR2022]'].notna().sum())

Unique value types: [<class 'float'>]
Number of numeric entries: 1257


In [5]:
# Step 2B: Pivot the dataset from long to wide

# Keep only the relevant columns
data_wide = data.pivot_table(
    index=['Country Name', 'Country Code'],
    columns='Series Name',
    values='2022 [YR2022]'
).reset_index()

# Flatten multi-level columns (if any)
data_wide.columns.name = None

print("✅ Reshaped dataset successfully.")
print("Shape after pivot:", data_wide.shape)
display(data_wide.head(5))

✅ Reshaped dataset successfully.
Shape after pivot: (264, 8)


,Country Name,Country Code,Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita),Energy use (kg of oil equivalent per capita),GDP per capita (constant 2015 US$),Renewable energy consumption (% of total final energy consumption),Trade (% of GDP),Urban population (% of total population)
0,Afghanistan,AFG,0.203552,NaN,377.665627,20.0,72.885470,26.616000
1,Africa Eastern and Southern,AFE,0.816361,562.060699,1421.797169,NaN,58.642287,37.909012
2,Africa Western and Central,AFW,0.505462,351.913062,1824.953296,NaN,NaN,49.129808
3,Albania,ALB,1.659293,780.738624,5178.884315,NaN,84.698057,63.799000
4,Algeria,DZA,4.104114,1499.510000,4544.466881,NaN,51.202376,74.772000


In [6]:
rename_map = {
    'Carbon dioxide (CO2) emissions excluding LULUCF per capita (t CO2e/capita)': 'CO2_PC',
    'GDP per capita (constant 2015 US$)': 'GDP_PC',
    'Energy use (kg of oil equivalent per capita)': 'Energy_PC',
    'Urban population (% of total population)': 'Urban_PCT',
    'Trade (% of GDP)': 'Trade_PCT',
    'Renewable energy consumption (% of total final energy consumption)': 'Renewable_PCT'
}

In [7]:
data_wide = data_wide.rename(columns=rename_map)

# Ensure numeric types (handles cases where missing values persisted)
num_cols = ['CO2_PC', 'GDP_PC', 'Energy_PC', 'Urban_PCT', 'Trade_PCT', 'Renewable_PCT']
data_wide[num_cols] = data_wide[num_cols].apply(pd.to_numeric, errors='coerce')

# Drop rows missing key variables
data_wide = data_wide.dropna(subset=['CO2_PC', 'GDP_PC', 'Energy_PC'])

print("✅ Cleaned dataset ready for transformations.")
print("Shape after cleaning:", data_wide.shape)
display(data_wide.head(5))

✅ Cleaned dataset ready for transformations.
Shape after cleaning: (184, 8)


,Country Name,Country Code,CO2_PC,Energy_PC,GDP_PC,Renewable_PCT,Trade_PCT,Urban_PCT
1,Africa Eastern and Southern,AFE,0.816361,562.060699,1421.797169,NaN,58.642287,37.909012
2,Africa Western and Central,AFW,0.505462,351.913062,1824.953296,NaN,NaN,49.129808
3,Albania,ALB,1.659293,780.738624,5178.884315,NaN,84.698057,63.799000
4,Algeria,DZA,4.104114,1499.510000,4544.466881,NaN,51.202376,74.772000
7,Angola,AGO,0.767587,423.122518,2382.022640,NaN,69.691071,68.081000


In [8]:
# Step 3A: Variable Transformations

# Apply natural logs (skip NaN safely)
data_wide['lnCO2'] = np.log(data_wide['CO2_PC'])
data_wide['lnGDP'] = np.log(data_wide['GDP_PC'])
data_wide['lnGDP_sq'] = data_wide['lnGDP'] ** 2
data_wide['lnEnergy'] = np.log(data_wide['Energy_PC'])
data_wide['lnUrban'] = np.log(data_wide['Urban_PCT'])
data_wide['lnTrade'] = np.log(data_wide['Trade_PCT'])

# Keep Renewable as is
data_wide['Renewable'] = data_wide['Renewable_PCT']

print("✅ Created transformed variables.")
display(data_wide[['Country Name', 'lnCO2', 'lnGDP', 'lnGDP_sq', 'lnEnergy', 'lnUrban', 'lnTrade', 'Renewable']].head(5))


✅ Created transformed variables.


,Country Name,lnCO2,lnGDP,lnGDP_sq,lnEnergy,lnUrban,lnTrade,Renewable
1,Africa Eastern and Southern,-0.202898,7.259677,52.702910,6.331610,3.635189,4.071456,NaN
2,Africa Western and Central,-0.682282,7.509310,56.389732,5.863384,3.894466,NaN,NaN
3,Albania,0.506392,8.552345,73.142604,6.660240,4.155738,4.439093,NaN
4,Algeria,1.411990,8.421666,70.924453,7.312894,4.314443,3.935786,NaN
7,Angola,-0.264503,7.775705,60.461592,6.047662,4.220698,4.244072,NaN


In [9]:
# Step 3B: Check sample size and missing data in regression variables

reg_vars = ['lnCO2','lnGDP','lnGDP_sq','lnEnergy','lnUrban','lnTrade','Renewable']
missing_summary = data_wide[reg_vars].isna().sum()

print("Rows in full dataset:", len(data_wide))
print("Rows available for regression:", len(data_wide.dropna(subset=reg_vars)))
print("\nMissing values per variable:")
print(missing_summary)

Rows in full dataset: 184
Rows available for regression: 7

Missing values per variable:
lnCO2          0
lnGDP          0
lnGDP_sq       0
lnEnergy       0
lnUrban        0
lnTrade       10
Renewable    176
dtype: int64


In [10]:
# Step 4 (Revised): Regression without Renewable

# Drop Renewable to retain more rows
reg_data = data_wide.dropna(subset=['lnCO2','lnGDP','lnGDP_sq','lnEnergy','lnUrban','lnTrade'])

# Define X and y
X = reg_data[['lnGDP','lnGDP_sq','lnEnergy','lnUrban','lnTrade']]
y = reg_data['lnCO2']

# Add constant
X = sm.add_constant(X)

# Fit model
model = sm.OLS(y, X).fit()

print("✅ Regression completed (without Renewable). Summary below:")
print(model.summary())

✅ Regression completed (without Renewable). Summary below:
                            OLS Regression Results                            
Dep. Variable:                  lnCO2   R-squared:                       0.913
Model:                            OLS   Adj. R-squared:                  0.911
Method:                 Least Squares   F-statistic:                     353.5
Date:                Fri, 31 Oct 2025   Prob (F-statistic):           3.55e-87
Time:                        23:39:42   Log-Likelihood:                -74.336
No. Observations:                 174   AIC:                             160.7
Df Residuals:                     168   BIC:                             179.6
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------

In [11]:
# Step 5: Multicollinearity & EKC Turning Point

from statsmodels.stats.outliers_influence import variance_inflation_factor

# 5A. Calculate VIF for each regressor
vif_df = pd.DataFrame()
vif_df["Variable"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
display(vif_df)

# 5B. Compute the EKC turning point in GDP per capita
b1 = model.params['lnGDP']
b2 = model.params['lnGDP_sq']

# ln(GDP_turn) = -β1 / (2β2)
lnGDP_turn = -b1 / (2 * b2)
GDP_turn = np.exp(lnGDP_turn)

print(f"\nEstimated turning point (ln scale): {lnGDP_turn:.3f}")
print(f"Estimated turning point GDP per capita: ${GDP_turn:,.0f} (2015 USD)")


,Variable,VIF
0,const,1793.501809
1,lnGDP,200.160021
2,lnGDP_sq,181.147851
3,lnEnergy,4.635752
4,lnUrban,2.523962
5,lnTrade,1.218522



Estimated turning point (ln scale): 9.471
Estimated turning point GDP per capita: $12,972 (2015 USD)
